In [1]:
# Inventory Allocation Optimizer - Notebook Version (Functions Only)
import os
import pandas as pd
import logging
from datetime import datetime

from database_connector import DatabaseConnector
from sql_query_loader import SQLQueryLoader
import config_loader
from data_processor import (
    process_demand_data, process_inventory_data, process_open_po_data,
    process_inbound_data, process_master_data, process_vendor_data, process_gfl_data
)
from calculations import calculate_all

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

start_time = datetime.now()

In [2]:
# Connection details (read from environment variables if set)
conn_params = {
    'user': os.getenv('REDSHIFT_USER', 'manh.nguyen@razor-group.com'),
    'password': os.getenv('REDSHIFT_PASSWORD', 'qdkcTHB8CPfe7AQHVNEF'),
    'database': os.getenv('REDSHIFT_DATABASE', 'dev'),
    'host': os.getenv('REDSHIFT_HOST', 'datawarehouse-dev.cdg4y3yokxle.eu-central-1.redshift.amazonaws.com'),
    'port': int(os.getenv('REDSHIFT_PORT', 5439))
}

In [3]:
# Connect to database
db = DatabaseConnector(
    host=conn_params['host'],
    dbname=conn_params['database'],
    user=conn_params['user'],
    password=conn_params['password'],
    port=conn_params['port']
)

# Load SQL query loader
query_loader = SQLQueryLoader()
db.set_query_loader(query_loader)

2025-08-10 02:54:07,356 - sql_query_loader - INFO - Loaded query: asin_vendor from asin_vendor_mapping.sql
2025-08-10 02:54:07,358 - sql_query_loader - INFO - Loaded query: target_sp from target_sales_price.sql
2025-08-10 02:54:07,358 - sql_query_loader - INFO - Loaded query: demand from demand_forecast.sql
2025-08-10 02:54:07,359 - sql_query_loader - INFO - Loaded query: master_data from master_data.sql
2025-08-10 02:54:07,359 - sql_query_loader - INFO - Loaded query: gfl_list from gfl_list.sql
2025-08-10 02:54:07,360 - sql_query_loader - INFO - Loaded query: vendor_master from vendor_master.sql
2025-08-10 02:54:07,360 - sql_query_loader - INFO - Loaded query: open_po from open_po.sql
2025-08-10 02:54:07,361 - sql_query_loader - INFO - Loaded query: otif_status from otif_status.sql
2025-08-10 02:54:07,361 - sql_query_loader - INFO - Loaded query: inbound from inbound_shipments.sql
2025-08-10 02:54:07,361 - sql_query_loader - INFO - Loaded query: inventory from inventory_sop.sql


In [4]:
# Define required queries
required_queries = [
    'asin_vendor', 'target_sp', 'demand', 'master_data',
    'gfl_list', 'vendor_master', 'open_po',
    'otif_status', 'inbound', 'inventory'
]

logger.info(f"Loading {len(required_queries)} queries in parallel...")
data = db.load_queries_parallel(required_queries, max_workers=5)

2025-08-10 02:54:07,366 - __main__ - INFO - Loading 10 queries in parallel...
/Users/teq-admin/Downloads/inventory-allocation-optimizer-1/database_connector.py:76: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
2025-08-10 02:54:09,892 - database_connector - ERROR - Error in query target_sp: Execution failed on sql '-- Target Sales Price Query
-- Fetches target sales prices for revenue calculations
-- Sources: rgbit_asp_l30_all_channels_w_fallbacks, forecast_reporting.system_forecast_reporting

WITH filtered_data AS (
    SELECT asin, sales_channel, marketplace, gross_asp_l30
    FROM razor_db.public.rgbit_asp_l30_all_channels_w_fallbacks
    WHERE gross_asp_l30 IS NOT NULL AND gross_asp_l30 > 0
),
aggregated_data AS (
    SELECT asin, marketplace, 
           LISTAGG(sales_channel, ', ') WITHIN GROUP (O

In [5]:
data['inventory']

,asin,mp,brand_name,in_fm,in_to_w2w,in_nn3pl,in_n3pl,in_lm,in_amz,units_in_d2amz,...,in_rsd_fc_proc,in_rsvd_cust_ord,in_walmart,in_bol,in_meli,in_to_osc_l3m,total_e2e_inventory,total_inventory,units_open_pos_raw,razin_list
0,B0002A9SJ2,US,Prosumers Choice,0.0,0.0,0.0,4603,23.0,279.0,0.0,...,0,2,0,0.0,0,48.0,4907.0,4907.0,0.0,PRCH-000080
1,B003SE2HOM,US,PowrX,0.0,0.0,0.0,0,0.0,0.0,0.0,...,14,0,0,0.0,0,0.0,14.0,14.0,0.0,POWR-000434
2,B007CMFUZ4,US,iPrimio,0.0,0.0,0.0,725,115.0,374.0,240.0,...,2,0,0,0.0,0,0.0,1456.0,1216.0,240.0,PXIP-000042
3,B007CMH9B2,US,iPrimio,0.0,0.0,0.0,1100,300.0,206.0,0.0,...,1,0,39,0.0,0,0.0,1607.0,1607.0,0.0,PXIP-000041
4,B008JEE5J6,EU,PowrX,0.0,0.0,0.0,1004,0.0,162.0,0.0,...,0,0,0,15.0,0,0.0,1181.0,1181.0,0.0,POWR-000305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14639,WIGA-100181,US,Window Garden,0.0,0.0,0.0,919,0.0,0.0,0.0,...,0,0,0,0.0,0,0.0,919.0,919.0,0.0,WIGA-100181
14640,WIGA-100182,US,Window Garden,0.0,0.0,0.0,1000,0.0,0.0,0.0,...,0,0,0,0.0,0,0.0,1000.0,1000.0,0.0,WIGA-100182
14641,WIGA-100198,US,Window Garden,0.0,0.0,0.0,909,0.0,0.0,0.0,...,0,0,0,0.0,0,0.0,909.0,909.0,0.0,WIGA-100198
14642,WIGA-100202,US,Window Garden,0.0,0.0,0.0,3272,0.0,0.0,0.0,...,0,0,0,0.0,0,0.0,3272.0,3272.0,0.0,WIGA-100202


In [6]:
# Process demand and inventory data
processed_data = {}

processed_data['dim_demand'] = (
    process_demand_data(data['demand']) if not data['demand'].empty else pd.DataFrame()
)

processed_data['dim_inventory'] = (
    process_inventory_data(data['inventory']) if not data['inventory'].empty else pd.DataFrame()
)

2025-08-10 02:54:28,997 - data_processor - INFO - Processing demand data: 346536 rows
2025-08-10 02:54:29,235 - data_processor - INFO - Processing inventory data: 14644 rows


In [7]:
# Process master data and open purchase orders
processed_data['dim_master'] = (
    process_master_data(data['master_data']) if not data['master_data'].empty else pd.DataFrame()
)

if not data['open_po'].empty:
    dim_open_po_signed, dim_open_po_unsigned = process_open_po_data(
        data['open_po'], processed_data['dim_master']
    )
else:
    dim_open_po_signed = pd.DataFrame()
    dim_open_po_unsigned = pd.DataFrame()

processed_data['dim_open_po_signed'] = dim_open_po_signed
processed_data['dim_open_po_unsigned'] = dim_open_po_unsigned

In [8]:
# Process inbound, vendor and target SP data
processed_data['dim_inbound'] = (
    process_inbound_data(data['inbound']) if not data['inbound'].empty else pd.DataFrame()
)

processed_data['dim_vendor'] = (
    process_vendor_data(data['vendor_master']) if not data['vendor_master'].empty else pd.DataFrame()
)

if 'target_sp' in data and not data['target_sp'].empty:
    if 'ref' in data['target_sp'].columns:
        processed_data['dim_target_sp'] = data['target_sp'].set_index('ref')
    else:
        processed_data['dim_target_sp'] = data['target_sp']
else:
    processed_data['dim_target_sp'] = pd.DataFrame()

2025-08-10 02:54:29,268 - data_processor - INFO - Processing inbound data: 3493 rows


In [9]:
# Process GFL list data
processed_data['dim_gfl'] = (
    process_gfl_data(data['gfl_list']) if not data['gfl_list'].empty else pd.DataFrame()
)

/Users/teq-admin/Downloads/inventory-allocation-optimizer-1/data_processor.py:450: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df['mp'] = df['marketplace'].replace('Pan-EU', 'EU')


In [10]:
# import pandas as pd

# def inspect_categorical_columns(df, name):
#     print(f"\nInspecting DataFrame: {name}")
#     for col in df.columns:
#         if pd.api.types.is_categorical_dtype(df[col]):
#             print(f"  - Column '{col}' is Categorical")
#             print(f"    Categories: {df[col].cat.categories.tolist()}")
#             print(f"    Contains 0 in categories? {0 in df[col].cat.categories}")
#     print("-" * 50)

# # Example: right before calculate_sales_missed call
# inspect_categorical_columns(dim_demand, "dim_demand")
# inspect_categorical_columns(dim_inventory, "dim_inventory")
# inspect_categorical_columns(dim_open_po_signed, "dim_open_po_signed")
# inspect_categorical_columns(dim_open_po_unsigned, "dim_open_po_unsigned")
# inspect_categorical_columns(dim_inbound, "dim_inbound")


In [11]:
# dim_demand = processed_data.get('dim_demand', pd.DataFrame())
# dim_inventory = processed_data.get('dim_inventory', pd.DataFrame())
# dim_open_po_signed = processed_data.get('dim_open_po_signed', pd.DataFrame())
# dim_open_po_unsigned = processed_data.get('dim_open_po_unsigned', pd.DataFrame())
# dim_inbound = processed_data.get('dim_inbound', pd.DataFrame())
# dim_target_sp = processed_data.get('dim_target_sp', pd.DataFrame())

In [12]:
# Run inventory allocation calculations
logger.info("Running inventory calculations...")
results = calculate_all(processed_data, config_loader)

2025-08-10 02:54:29,406 - __main__ - INFO - Running inventory calculations...
2025-08-10 02:54:29,408 - calculations - INFO - Starting comprehensive inventory calculations...
2025-08-10 02:54:29,410 - calculations - INFO - Starting sales missed calculations...
/Users/teq-admin/Downloads/inventory-allocation-optimizer-1/calculations.py:148: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dim_sales_missed['OOS_week_with_signed_final'] = dim_sales_missed['OOS_week_with_signed_last'].fillna(
2025-08-10 02:54:29,683 - calculations - INFO - Sales missed calculations completed
2025-08-10 02:54:29,684 - calculations - INFO - Calculating revenue impact...
2025-08-10 02:54:30,614 - calculations - INFO - Revenue impact calculations completed
2025-08-10 02:54:30,614 - ca

In [13]:
# Save results to CSV
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = f"{output_dir}/inventory_allocation_{timestamp}.csv"

if 'final_allocation' in results and not results['final_allocation'].empty:
    results['final_allocation'].to_csv(output_file, index=False)
    logger.info(f"Results saved to {output_file}")
else:
    logger.warning("No final allocation data to save")

2025-08-10 02:54:38,734 - __main__ - INFO - Results saved to output/inventory_allocation_20250810_025437.csv


In [14]:
# Print execution summary
elapsed_time = (datetime.now() - start_time).total_seconds()

print("\n" + "="*50)
print("INVENTORY ALLOCATION SUMMARY")
print("="*50)
print(f"Total SKUs processed: {len(results['final_allocation']) if 'final_allocation' in results and not results['final_allocation'].empty else 0}")
print(f"Demand coverage: {results.get('demand_coverage', 'N/A')}%")
print(f"Out of stock predictions: {results.get('oos_count', 0)} items")
print(f"Processing time: {elapsed_time:.2f} seconds")
print("="*50)


INVENTORY ALLOCATION SUMMARY
Total SKUs processed: 30905
Demand coverage: 100.0%
Out of stock predictions: 30905 items
Processing time: 31.39 seconds


In [15]:
# Close database connection
db.close()
logger.info("Database connection closed")

2025-08-10 02:54:38,745 - __main__ - INFO - Database connection closed
